In [28]:
from pyspark.sql import SparkSession

spark= SparkSession.builder\
       .appName("Bonus_JSON")\
       .getOrCreate()

sc= spark.sparkContext
print("Spark session created")

Spark session created


In [21]:
patients_csv = """patient_id,name,city,age,gender,registration_date
1,Aarav,Hyderabad,29,Male,2023-01-10
2,Priya,Bangalore,34,Female,2023-02-12
3,Rahul,Mumbai,41,Male,2023-03-14
4,Sneha,Delhi,26,Female,2023-04-15
5,Kiran,Chennai,37,Male,2023-05-11
6,Meera,Hyderabad,31,Female,2023-06-10
7,Amit,Pune,45,Male,2023-06-22
8,Neha,Delhi,28,Female,2023-07-10
9,Divya,Bangalore,33,Female,2023-07-15
10,Vikram,Mumbai,52,Male,2023-08-01
11,Farhan,Hyderabad,39,Male,2023-08-10
12,Simran,Delhi,25,Female,2023-08-21
"""

with open("patients.csv", "w") as f:
    f.write(patients_csv)
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
patients.show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [2]:
patient_profiles_json = """[
  {
    "patient_id": 1,
    "name": "Aarav",
    "contact": {"email": "aarav@mail.com", "phone": "9000011111"},
    "allergies": ["Dust", "Peanuts"]
  },
  {
    "patient_id": 2,
    "name": "Priya",
    "contact": {"email": "priya@mail.com", "phone": "9000022222"},
    "allergies": ["Pollen"]
  },
  {
    "patient_id": 3,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000033333"},
    "allergies": ["Dust", "Milk"]
  },
  {
    "patient_id": 6,
    "name": "Meera",
    "contact": {"email": "meera@mail.com", "phone": "9000066666"},
    "allergies": ["Seafood"]
  },
  {
    "patient_id": 10,
    "name": "Vikram",
    "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
    "allergies": ["Pollen", "Dust"]
  }
]
"""

with open("patient_profiles.json", "w") as f:
    f.write(patient_profiles_json)
profiles = spark.read.json("patient_profiles.json", multiLine=True)
profiles.show()

+---------------+--------------------+------+----------+
|      allergies|             contact|  name|patient_id|
+---------------+--------------------+------+----------+
|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav|         1|
|       [Pollen]|{priya@mail.com, ...| Priya|         2|
|   [Dust, Milk]|{rahul@mail.com, ...| Rahul|         3|
|      [Seafood]|{meera@mail.com, ...| Meera|         6|
| [Pollen, Dust]|{vikram@mail.com,...|Vikram|        10|
+---------------+--------------------+------+----------+



In [ ]:
#Exercise 1
profiles.show()

In [ ]:
#Exercise 2
profiles.printSchema()

In [ ]:
#Exercise 3
from pyspark.sql.functions import col, explode, struct, count
profiles.select("patient_id", col("name"), col("contact.email").alias("Email"), col("contact.phone").alias("phone")).show()

In [ ]:
#Exercise 4
profiles.select(explode(col("allergies")).alias("allergy")).show()

In [ ]:
#Exercise 5
profiles.select(explode(col("allergies")).alias("allergy")).groupby("allergy").agg(count("*").alias("patient_count")).show()

In [ ]:
#Exercise 6
profiles.select(col("patient_id"), col("name"),explode(col("allergies")).alias("allergy")).filter(col("allergy")=="Dust").select("patient_id", "name", "allergy").show()

In [ ]:
#Exercise 7
profiles.select(explode(col("allergies")).alias("allergy")).distinct().show()

In [ ]:
#Exercise 8
profiles.join(patients, on="patient_id", how="inner").show()

In [ ]:
#Exercise 9
profiles.alias("pr").join(patients.alias("pa"), "patient_id") \
    .select(
        col("pa.name").alias("patient_name"),
        col("pa.city"),
        col("pr.contact.email").alias("Email"),
        explode(col("pr.allergies")).alias("allergy")
    ).show()

In [ ]:
#Exercise 10
profiles.alias("pr").join(patients.alias("pa"), "patient_id") \
    .select(col("pa.city"),explode(col("pr.allergies")).alias("allergy")) \
    .groupBy("city", "allergy") \
    .agg(count("*").alias("allergy_count")) \
    .orderBy("city") \
    .show()